# Advanced Allergy Web Research with MCP

This notebook builds a small **agentic web-research workflow** with two MCP servers:

- **Playwright MCP**: browse trusted medical websites.
- **Filesystem MCP**: save the final research report to `allergy_advanced.md`.

The research focuses on **early allergy symptoms**, how symptoms differ by allergy type, **anaphylaxis warning signs**, and a practical **allergy vs flu** comparison. The agent is instructed to use multiple authoritative medical sources, synthesize them, save the report, and then stop.

> This is an educational research workflow, not a diagnostic tool.


## 1. Environment and Windows Notebook support

On native Windows Jupyter, `MCPServerStdio` needs a subprocess-capable event loop. The helper below runs MCP work on a Windows `ProactorEventLoop`. It also sends MCP stderr to `mcp_stderr.log`, which makes failures easier to diagnose.


In [1]:
import asyncio
import os
import sys
from contextlib import asynccontextmanager
from pathlib import Path

from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio

load_dotenv(override=True)
os.environ["NODE_USE_SYSTEM_CA"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"

import agents.mcp.server as agents_mcp_server
try:
    from mcp import stdio_client
except ImportError:
    from mcp.client.stdio import stdio_client

@asynccontextmanager
async def notebook_stdio(params):
    with open("mcp_stderr.log", "a", encoding="utf-8") as log:
        async with stdio_client(params, errlog=log) as streams:
            yield streams

agents_mcp_server.stdio_client = notebook_stdio

def run_on_proactor(async_fn):
    from asyncio.windows_events import ProactorEventLoop
    loop = ProactorEventLoop()
    asyncio.set_event_loop(loop)
    try:
        result = loop.run_until_complete(async_fn())
        loop.run_until_complete(asyncio.sleep(0.3))
        return result
    finally:
        asyncio.set_event_loop(None)
        loop.close()

async def run_mcp(async_fn):
    if sys.platform == "win32" and "ipykernel" in sys.modules:
        return await asyncio.to_thread(run_on_proactor, async_fn)
    return await async_fn()


## 2. Start the MCP servers

The Playwright server runs headless. `NODE_USE_SYSTEM_CA=1` is already set above so `npx` can use the Windows trusted certificate store. The filesystem server is restricted to the local `sandbox` folder.


In [2]:
sandbox_path = os.path.abspath("sandbox")
os.makedirs(sandbox_path, exist_ok=True)

playwright_params = {
    "command": "cmd",
    "args": ["/c", "npx", "-y", "@playwright/mcp@latest", "--headless"],
    "env": os.environ.copy(),
}

files_params = {
    "command": "cmd",
    "args": ["/c", "npx", "-y", "@modelcontextprotocol/server-filesystem", sandbox_path],
    "env": os.environ.copy(),
}


## 3. Research plan

Instead of stopping after a single page, this version performs a **focused multi-source review**. It prioritizes authoritative sources and stops after it has enough evidence to cover the requested topics.

Target evidence:

1. General allergy symptoms and common triggers.
2. Early respiratory-allergy / hay-fever symptoms and timing after exposure.
3. Food, medication, insect-sting, and skin-related symptoms.
4. Early and severe anaphylaxis warning signs and emergency actions.
5. Flu symptoms, onset, and emergency warning signs.
6. A clear allergy-vs-flu comparison table.

The agent should favor **Mayo Clinic, AAAAI, ACAAI, CDC, FDA, and NHS**, record source URLs, avoid unsupported claims, and stop browsing once these categories are covered.


In [3]:
instructions = """
You are a careful medical web-research assistant. This is an educational research task, not a diagnostic task.

GOAL
Create a comprehensive but concise report about early allergy symptoms and how allergy differs from influenza (flu).

SOURCE QUALITY
- Use 4 to 6 authoritative sources total.
- Prefer Mayo Clinic, AAAAI, ACAAI, CDC, FDA, or NHS.
- Use primary/official medical pages rather than blogs, forums, ads, or social media.
- Record the organization, page title, and URL for every source used.

RESEARCH TOPICS
A. Explain what an allergy is and note that symptoms vary by allergen and body system.
B. Identify early/common symptoms of allergic rhinitis/hay fever, especially itching, sneezing, watery eyes, runny nose, congestion, cough/postnasal drip, and timing after exposure when supported by sources.
C. Cover early symptoms of food allergy, medication allergy, insect-sting allergy, and skin allergy when reliable sources support them.
D. Separate mild/moderate allergy symptoms from severe allergic reaction/anaphylaxis.
E. For anaphylaxis, summarize timing, early/evolving warning signs, severe warning signs, and emergency actions. Do not delay emergency care while researching.
F. Use CDC material to summarize flu: usual abrupt onset, fever/chills, cough, sore throat, body aches, headache, fatigue, and other supported symptoms.
G. Build an ALLERGY VS FLU table covering: cause, contagiousness, onset, itching, sneezing, runny/stuffy nose, fever/chills, cough, sore throat, body aches, headache, fatigue, hives/swelling, GI symptoms, trigger/season pattern, duration pattern, and when testing/medical evaluation may be needed.
H. Add a short 'Practical clues' section. Clearly state that symptoms can overlap and the table cannot diagnose a person.

BROWSING STRATEGY
1. Start with these pages when accessible:
   - Mayo Clinic Allergies: https://www.mayoclinic.org/diseases-conditions/allergies/symptoms-causes/syc-20351497
   - Mayo Clinic Hay fever: https://www.mayoclinic.org/diseases-conditions/hay-fever/symptoms-causes/syc-20373039
   - Mayo Clinic Anaphylaxis: https://www.mayoclinic.org/diseases-conditions/anaphylaxis/symptoms-causes/syc-20351468
   - AAAAI symptom reference: https://allergist.aaaai.org/virtual-allergist/
   - CDC Flu symptoms: https://www.cdc.gov/flu/signs-symptoms/index.html
2. If a page is inaccessible, search for an equivalent official page from the same or another preferred organization.
3. Once every research topic above has adequate evidence, STOP browsing. Do not keep searching for more pages just to increase the source count.

OUTPUT FORMAT FOR allergy_advanced.md
# Advanced Allergy Research: Early Symptoms and Allergy vs Flu
## Key takeaway
## Early allergy symptoms by type
### Allergic rhinitis / hay fever
### Food allergy
### Insect-sting allergy
### Medication allergy
### Skin/contact allergy (only if supported)
## Early anaphylaxis warning pattern
## Emergency warning signs and what to do
## Allergy vs Flu
Use a markdown comparison table.
## Practical clues
## When to seek medical evaluation
## Sources

SAFETY AND ACCURACY
- Do not diagnose.
- Do not say that one symptom proves allergy or flu.
- Distinguish routine allergy symptoms from anaphylaxis.
- State that not everyone with flu has fever.
- If severe allergic reaction signs are present, advise emergency services and prescribed epinephrine according to the person's emergency plan, consistent with the source.
- Keep wording proportional to the evidence and avoid unsupported treatment recommendations.

FINISH
- Save the final report as allergy_advanced.md in the allowed filesystem directory.
- After the file is successfully written, STOP using tools.
- Return only a short confirmation with the file name and number of sources used.
"""

async def advanced_allergy_research():
    async with MCPServerStdio(
        name="filesystem",
        params=files_params,
        client_session_timeout_seconds=180,
    ) as files_server:
        async with MCPServerStdio(
            name="playwright",
            params=playwright_params,
            client_session_timeout_seconds=180,
        ) as browser_server:
            agent = Agent(
                name="advanced_allergy_researcher",
                instructions=instructions,
                model="gpt-4.1-mini",
                mcp_servers=[browser_server, files_server],
            )

            with trace("advanced_allergy_research"):
                result = await Runner.run(
                    agent,
                    "Research early allergy symptoms comprehensively, compare allergy with flu, and save the report to allergy_advanced.md.",
                    max_turns=35,
                )
                return result.final_output

final_output = await run_mcp(advanced_allergy_research)
print("Agent:", final_output)


Agent: Report completed and saved as allergy_advanced.md using 4 authoritative sources.


## 4. Display the saved research report

This final cell reads the markdown file written by the filesystem MCP server, so the research result appears directly in the notebook.


In [4]:
report_file = Path(sandbox_path) / "allergy_advanced.md"

if report_file.exists():
    report_text = report_file.read_text(encoding="utf-8")
    print(report_text)
else:
    print("allergy_advanced.md was not created. Check mcp_stderr.log for MCP errors.")


# Advanced Allergy Research: Early Symptoms and Allergy vs Flu

## Key takeaway
An allergy is an immune system overreaction to harmless substances called allergens. Symptoms vary by allergen and affected body system and can range from mild irritation to life-threatening anaphylaxis. Early allergy signs commonly include itching, sneezing, runny nose, watery eyes, and congestion, typically appearing shortly after allergen exposure. Severe allergic reactions (anaphylaxis) develop rapidly and require immediate emergency care. Influenza (flu) is a contagious viral illness with abrupt onset, fever, body aches, and respiratory symptoms that can overlap with allergy symptoms but is caused by infection, not an immune overreaction.

## Early allergy symptoms by type

### Allergic rhinitis / hay fever
- Immune response to allergens like pollen, dust mites, animal dander, mold.
- Common early symptoms: itching in nose, roof of mouth, throat, eyes; frequent sneezing; runny or blocked nose; watery, 